In [1]:
#!python -m spacy download es_core_news_md
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
from unidecode import unidecode
import  pandas as pd
import spacy
import nltk
import re

Cargar base de datos
Conjunto de datos de reseñas de restaurantes de la ciudad de barcelona  sacados de la 
aplicacion de TripAdvisor la cual es usada para recomendar restaurantes a los turistas o a los locales

In [2]:
#D:\Estudios\PLN\MODULO 1\Actividad 2\Tarea2\data
data = pd.read_csv('D:/Estudios/PLN/MODULO 1/Actividad 2/Tarea2/data/Barcelona_reviews.csv')


C:\Users\carol\AppData\Local\Temp\ipykernel_46000\3909773870.py:2: DtypeWarning: Columns (0: Unnamed: 0, 1: parse_count, 2: rating_review) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('D:/Estudios/PLN/MODULO 1/Actividad 2/Tarea2/data/Barcelona_reviews.csv')


In [3]:
data

,Unnamed: 0,parse_count,restaurant_name,rating_review,sample,review_id,title_review,review_preview,review_full,date,city,url_restaurant,author_id
0,0,1,Chalito_Rambla,1,Negative,review_774086112,Terrible food Terrible service,"Ok, this place is terrible! Came here bc we’ve...","Ok, this place is terrible! Came here bc we’ve...","October 12, 2020",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_0
1,1,2,Chalito_Rambla,5,Positive,review_739142140,The best milanesa in central Barcelona,This place was a great surprise. The food is d...,This place was a great surprise. The food is d...,"January 14, 2020",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_1
2,2,3,Chalito_Rambla,5,Positive,review_749758638,Family bonding,The food is excellent.....the ambiance is very...,The food is excellent.....the ambiance is very...,"March 7, 2020",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_2
3,3,4,Chalito_Rambla,5,Positive,review_749732001,Best food,"The food is execellent ,affortable price for p...","The food is execellent ,affortable price for p...","March 7, 2020",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_3
4,4,5,Chalito_Rambla,5,Positive,review_749691057,Amazing Food and Fantastic Service,"Mr Suarez,The food at your restaurant was abso...","Mr Suarez,The food at your restaurant was abso...","March 7, 2020",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
416351,426638,426639,Bodega_Biarritz,5,Positive,review_656474442,Awesome spot!,What a cute little hole in wall! The decorum w...,What a cute little hole in wall! The decorum w...,"March 4, 2019",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_232019
416352,426639,426640,Bodega_Biarritz,5,Positive,review_656066564,Great tapas and great service,We really enjoyed the tapas here. Each of the...,We really enjoyed the tapas here. Each of the ...,"March 3, 2019",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_57970
416353,426640,426641,Bodega_Biarritz,5,Positive,review_631216398,Fabulous Tapas,Tried to get a table at Bodega Biarritz 1820 b...,Tried to get a table at Bodega Biarritz 1820 b...,"November 6, 2018",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_36360
416354,426641,426642,Bodega_Biarritz,5,Positive,review_630522900,Fantastic!,It was our first time eating tapas and it was ...,It was our first time eating tapas and it was ...,"November 3, 2018",Barcelona_Catalonia,https://www.tripadvisor.com/Restaurant_Review-...,UID_18363


In [4]:
#Tamaño del conjunto de datos
data.shape

(416356, 13)

In [5]:
#Limpiar los textos de las reseñas : Borrar espacios extra, saltos de línea, tabs, etc.
data['review_preview'] = data['review_preview'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [6]:
data['sample'].unique()

<StringArray>
[                                                                                                                         'Negative',
                                                                                                                          'Positive',
 'https://www.tripadvisor.com/Restaurant_Review-g187497-d1081809-Reviews-or170-Arrosseria_Xativa_Les_Corts-Barcelona_Catalonia.html']
Length: 3, dtype: str

In [7]:
#Cambiar de formato las etiquetas de clasificación : Negativo = 0, Positivo = 1
#Eliminando los registros con etiquetas no definidas
data['target'] = data['sample'].map({
    'Negative': 0,
    'Positive': 1
})

data = data.dropna(subset=['target'])

data['target'] = data['target'].astype(int)

In [8]:
data[['target',"review_preview" , "sample"]].head(10)

,target,review_preview,sample
0,0,"Ok, this place is terrible! Came here bc we’ve...",Negative
1,1,This place was a great surprise. The food is d...,Positive
2,1,The food is excellent.....the ambiance is very...,Positive
3,1,"The food is execellent ,affortable price for p...",Positive
4,1,"Mr Suarez,The food at your restaurant was abso...",Positive
5,1,"LEAN Excellent environment, friendly service, ...",Positive
6,1,"LEAN.Excellent environment, friendly service, ...",Positive
7,1,Excellent food and lovely atmosphere! The wait...,Positive
8,1,I visited this placed yesterday to catch a qui...,Positive
9,1,The food here is sooooo good!! So good and tas...,Positive


In [9]:
#Mostrar el tamaño del conjunto de datos después de la limpieza
#solo se elimino un registro con etiqueta no definida
data.shape

(416355, 14)

In [10]:
#Revisar la distribución de las clases en el conjunto de datos
class_dist = data['target'].value_counts()
print(class_dist)

target
1    338779
0     77576
Name: count, dtype: int64


In [11]:
# Cargar modelo spaCy
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

# Stopwords en inglés
stopwords_en = set(stopwords.words("english"))

# Mantener negaciones
neg_keep = {"not", "no", "nor", "never"}

stopwords_en = stopwords_en - neg_keep

In [12]:


# Regex para signos
punct_re = re.compile(r"[()\[\]{}\/\\'¿!¡?.,;:\"<>|@#$%^&*_+=~`-]")

# Procesamiento masivo eficiente
docs = nlp.pipe(
    data['review_preview'],
    batch_size=1000
)


# Función para limpiar tokens
def clean_doc(doc):

    tokens = []

    for t in doc:

        # Eliminar espacios, signos y números
        if t.is_space or t.is_punct or t.is_digit:
            continue

        # Lematización y minúsculas
        lemma = t.lemma_.lower()

        # Eliminar stopwords
        if lemma in stopwords_en:
            continue

        # Quitar acentos
        clean_tok = unidecode(lemma)

        # Eliminar signos restantes
        clean_tok = punct_re.sub("", clean_tok)

        # Mantener solo texto válido
        if clean_tok and clean_tok.isalpha():
            tokens.append(clean_tok)

    return tokens

# Crear nueva columna tokenizada
data['tokenized_text'] = [clean_doc(doc) for doc in docs]

In [13]:
data[['sample','target',"review_preview" ,'tokenized_text']].head(10)




,sample,target,review_preview,tokenized_text
0,Negative,0,"Ok, this place is terrible! Came here bc we’ve...","[ok, place, terrible, come, bc, ve, always, wa..."
1,Positive,1,This place was a great surprise. The food is d...,"[place, great, surprise, food, delicious, dess..."
2,Positive,1,The food is excellent.....the ambiance is very...,"[food, excellent, ambiance, nice, price, affor..."
3,Positive,1,"The food is execellent ,affortable price for p...","[food, execellent, affortable, price, people, ..."
4,Positive,1,"Mr Suarez,The food at your restaurant was abso...","[mr, suarez, food, restaurant, absolutely, ama..."
5,Positive,1,"LEAN Excellent environment, friendly service, ...","[lean, excellent, environment, friendly, servi..."
6,Positive,1,"LEAN.Excellent environment, friendly service, ...","[leanexcellent, environment, friendly, service..."
7,Positive,1,Excellent food and lovely atmosphere! The wait...,"[excellent, food, lovely, atmosphere, waitress..."
8,Positive,1,I visited this placed yesterday to catch a qui...,"[visit, place, yesterday, catch, quick, bite, ..."
9,Positive,1,The food here is sooooo good!! So good and tas...,"[food, sooooo, good, good, tasty, yaaaam, good..."


In [114]:
token_frequencies = data['tokenized_text'].explode()
token_frequencies = token_frequencies.value_counts().reset_index()
token_frequencies.columns = ['token', 'frequency']
token_frequencies_gt20 = token_frequencies[token_frequencies['frequency'] > 10]
vocab = token_frequencies_gt20['token'].to_list()
data['tokenized_text_without_vocab'] = data['tokenized_text'].apply(lambda tokens: [t for t in tokens if t in vocab])

In [115]:
data[['sample','target',"review_preview" ,'tokenized_text','tokenized_text_without_vocab']].head(10)

,sample,target,review_preview,tokenized_text,tokenized_text_without_vocab
0,Negative,0,"Ok, this place is terrible! Came here bc we’ve...","[ok, place, terrible, come, bc, ve, always, wa...","[ok, place, terrible, come, bc, ve, always, wa..."
1,Positive,1,This place was a great surprise. The food is d...,"[place, great, surprise, food, delicious, dess...","[place, great, surprise, food, delicious, dess..."
2,Positive,1,The food is excellent.....the ambiance is very...,"[food, excellent, ambiance, nice, price, affor...","[food, excellent, ambiance, nice, price, affor..."
3,Positive,1,"The food is execellent ,affortable price for p...","[food, execellent, affortable, price, people, ...","[food, execellent, affortable, price, people, ..."
4,Positive,1,"Mr Suarez,The food at your restaurant was abso...","[mr, suarez, food, restaurant, absolutely, ama...","[mr, food, restaurant, absolutely, amazing, ho..."
5,Positive,1,"LEAN Excellent environment, friendly service, ...","[lean, excellent, environment, friendly, servi...","[lean, excellent, environment, friendly, servi..."
6,Positive,1,"LEAN.Excellent environment, friendly service, ...","[leanexcellent, environment, friendly, service...","[environment, friendly, service, great, menu, ..."
7,Positive,1,Excellent food and lovely atmosphere! The wait...,"[excellent, food, lovely, atmosphere, waitress...","[excellent, food, lovely, atmosphere, waitress..."
8,Positive,1,I visited this placed yesterday to catch a qui...,"[visit, place, yesterday, catch, quick, bite, ...","[visit, place, yesterday, catch, quick, bite, ..."
9,Positive,1,The food here is sooooo good!! So good and tas...,"[food, sooooo, good, good, tasty, yaaaam, good...","[food, sooooo, good, good, tasty, good, sangri..."


In [117]:
# Número de tokens original
len_original = len(data['tokenized_text'].iloc[100])

# Número de tokens  después del filtrado
len_filtrado = len(data['tokenized_text_without_vocab'].iloc[0])

print("Tokens originales:", len_original)
print("Tokens filtrados:", len_filtrado)

Tokens originales: 26
Tokens filtrados: 24


In [118]:

#Se crea una columna uniendo los tokens
data['clean_text'] = data['tokenized_text_without_vocab'].apply(
    lambda tokens: " ".join(tokens)
)

In [119]:
data[['sample','target',"review_preview" ,'tokenized_text','clean_text']].head(10)

,sample,target,review_preview,tokenized_text,clean_text
0,Negative,0,"Ok, this place is terrible! Came here bc we’ve...","[ok, place, terrible, come, bc, ve, always, wa...",ok place terrible come bc ve always walk aroun...
1,Positive,1,This place was a great surprise. The food is d...,"[place, great, surprise, food, delicious, dess...",place great surprise food delicious dessert we...
2,Positive,1,The food is excellent.....the ambiance is very...,"[food, excellent, ambiance, nice, price, affor...",food excellent ambiance nice price affordable ...
3,Positive,1,"The food is execellent ,affortable price for p...","[food, execellent, affortable, price, people, ...",food execellent affortable price people friend...
4,Positive,1,"Mr Suarez,The food at your restaurant was abso...","[mr, suarez, food, restaurant, absolutely, ama...",mr food restaurant absolutely amazing however ...
5,Positive,1,"LEAN Excellent environment, friendly service, ...","[lean, excellent, environment, friendly, servi...",lean excellent environment friendly service gr...
6,Positive,1,"LEAN.Excellent environment, friendly service, ...","[leanexcellent, environment, friendly, service...",environment friendly service great menu choice...
7,Positive,1,Excellent food and lovely atmosphere! The wait...,"[excellent, food, lovely, atmosphere, waitress...",excellent food lovely atmosphere waitress real...
8,Positive,1,I visited this placed yesterday to catch a qui...,"[visit, place, yesterday, catch, quick, bite, ...",visit place yesterday catch quick bite head ai...
9,Positive,1,The food here is sooooo good!! So good and tas...,"[food, sooooo, good, good, tasty, yaaaam, good...",food sooooo good good tasty good sangria try s...


In [120]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=10,
    max_features=5000
)

X = tfidf.fit_transform(data['clean_text'])

In [121]:
print(X.shape)

(416355, 5000)


In [122]:
#Asignacion de variables y separacion del conjunto de datos de entrenamiento y prueba
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    data['target'],
    test_size=0.2,
    random_state=42,
    stratify=data['target']
)

In [123]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
     class_weight='balanced',
     random_state=42,
     max_iter=5000
)

svm_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",'balanced'
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseud

In [124]:
y_pred = svm_model.predict(X_test)

In [125]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_test, y_pred))

0.8768598911986165


In [126]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.86      0.72     15515
           1       0.96      0.88      0.92     67756

    accuracy                           0.88     83271
   macro avg       0.79      0.87      0.82     83271
weighted avg       0.90      0.88      0.88     83271



In [127]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[13315  2200]
 [ 8054 59702]]


SEPARACION DE CONJUNTO , UNDERSAMPLING Y VECTORIZACION 

In [128]:

from sklearn.model_selection import train_test_split
#80% para entrenamiento y 20% para prueba, con estratificación para mantener la proporción de clases

X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    data['clean_text'],
    data['target'],
    test_size=0.2,
    stratify=data['target'],
    random_state=42
)


In [129]:
#se crea un dataframe para entrenar 
#con un conjunto de datos balanceado
# 2. Crear dataframe train
# Crear dataframe temporal SOLO para train
train_df = pd.DataFrame({
    'clean_text': X_train_bal,
    'target': y_train_bal
})

# separar clases
pos = train_df[train_df['target'] == 1]
neg = train_df[train_df['target'] == 0]

# undersampling positivos
pos_sample = pos.sample(
    n=len(neg),
    random_state=42
)

# dataset balanceado
train_bal = pd.concat(
    [pos_sample, neg]
).sample(frac=1, random_state=42)

# recuperar X e y
X_train_bal = train_bal['clean_text']
y_train_bal = train_bal['target']

In [130]:
X_train_bal.shape, y_train_bal.shape

((124122,), (124122,))

In [131]:
train_bal

,clean_text,target
344881,come across organic market accident day sights...,1
315156,food usually good price lunch menu around area...,1
232308,food service excellent order sopa de pescado a...,1
57531,friend walk past one guy ask say look offer tr...,1
317401,go travel bar interactive cooking class tell s...,0
...,...,...
126925,restaurant serve combination japanese peruvian...,0
364052,would hear great thing tiny tapas place always...,0
119103,take local foodie really not know expect every...,1
193964,not really big fan irish pub place good atmosp...,1


In [132]:
conteo = y_train_bal.value_counts().sort_index()

print("Negativos (0):", conteo[0])
print("Positivos (1):", conteo[1])

Negativos (0): 62061
Positivos (1): 62061


VECTORIZACION

In [133]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train_bal) # vectorizar el conjunto de entrenamiento balanceado

X_test_tfidf = vectorizer.transform(X_test_bal) # vectorizar el conjunto de prueba 

In [134]:
#416355 documentos (reviews) y 110141 características (tokens únicos o bigramas uniccos)
print(X_train_tfidf.shape)

(124122, 5000)


Modelos

SVM

In [135]:
from sklearn.svm import LinearSVC

modelo = LinearSVC(
    random_state=42,
    max_iter=3000
)

modelo.fit(
    X_train_tfidf,
    y_train_bal
)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [136]:
y_pred_bal = modelo.predict(X_test_tfidf)


In [137]:
from sklearn.metrics import accuracy_score

print(accuracy_score(y_test_bal, y_pred_bal))

0.8617045550071454


In [138]:
from sklearn.metrics import classification_report

print(classification_report(y_test_bal, y_pred_bal))

              precision    recall  f1-score   support

           0       0.59      0.85      0.70     15515
           1       0.96      0.86      0.91     67756

    accuracy                           0.86     83271
   macro avg       0.78      0.86      0.80     83271
weighted avg       0.89      0.86      0.87     83271



In [139]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_bal, y_pred_bal)

print(cm)

[[13162  2353]
 [ 9163 58593]]


nkjbjhbhj

Guardar el modelo que presento mejores resultados

In [141]:
import joblib

# guardar modelo
joblib.dump(
    svm_model,
    "svm_model.pkl"
)



['svm_model.pkl']

In [154]:
# guardar tfidf vectorizador
joblib.dump(
    tfidf,
    "tfidf_vectorizer.pkl"
)

['tfidf_vectorizer.pkl']

In [157]:
#Mostrar si se guardaron los archivos correctamente
import os  

print(os.listdir())

['Fase 1.docx', 'Fase1.ipynb', 'svm_model.pkl', 'tfidf_vectorizer.pkl', '~$Fase 1.docx', '~WRL3594.tmp']


In [158]:
#cargar el modelo y el vectorizador para futuras predicciones

modelo = joblib.load(
    "svm_model.pkl"
)

vectorizer = joblib.load(
    "tfidf_vectorizer.pkl"
)

In [160]:
pred = modelo.predict(
    vectorizer.transform(
        ["Texto Negativo =The days at the hotel were very bad, the people were very rude and all the hotel furniture was very dirty."]
    )
)

print(pred)

[0]
